$$\text{Modelo NMF - prueba}$$

# Importar librerías

In [ ]:
# Libraries imports
## previous modules import

import os
import sys
import warnings
from pathlib import Path

import pandas as pd
import numpy as np


print("\u2705 All pipeline modules imported successfully.")

In [ ]:
## NMF modeling specific imports

from time import time

import matplotlib.pyplot as plt

from sklearn.decomposition import NMF, MiniBatchNMF
from sklearn.feature_extraction.text import TfidfVectorizer

print("\u2705 All NMF modules imported successfully.")

# Traer los datos

> **paths**

In [ ]:
def _find_project_root(marker: str = "pyproject.toml") -> Path:
    """Walk up from cwd until a directory containing *marker* is found."""
    for candidate in [Path.cwd().resolve(), *Path.cwd().resolve().parents]:
        if (candidate / marker).exists():
            return candidate
    raise FileNotFoundError(
        f"Could not find '{marker}' in any parent of {Path.cwd().resolve()}.\n"
        "Make sure pyproject.toml exists at the project root."
    )

_PROJECT_ROOT     = _find_project_root()
_DATA = _PROJECT_ROOT / "data"
EXCEL_PATH = _DATA / 'processed' / 'obs20_nostopwords_noadverbs.xlsx'

FIGURE_PATH = _PROJECT_ROOT / "figures" / "nmf"

In [ ]:
data = pd.read_excel(EXCEL_PATH, sheet_name='obs20', engine='openpyxl')

In [ ]:
data.info()

# Función para graficar y guardar imágenes

In [ ]:
# # Plotting function for NMF topics
# def plot_top_words(model, feature_names, n_top_words, title, save_dir = FIGURE_PATH, filename= None):
#     fig, axes = plt.subplots(2, 5, figsize=(30, 15), sharex=True)
#     axes = axes.flatten()
#     for topic_idx, topic in enumerate(model.components_):
#         top_features_ind = topic.argsort()[-n_top_words:]
#         top_features = feature_names[top_features_ind]
#         weights = topic[top_features_ind]

#         ax = axes[topic_idx]
#         ax.barh(top_features, weights, height=0.7)
#         ax.set_title(f"Topic {topic_idx + 1}", fontdict={"fontsize": 30})
#         ax.tick_params(axis="both", which="major", labelsize=20)
#         for i in "top right left".split():
#             ax.spines[i].set_visible(False)
#         fig.suptitle(title, fontsize=40)

#     plt.subplots_adjust(top=0.90, bottom=0.05, wspace=0.90, hspace=0.3)

#     save_path = save_dir / filename

#     plt.savefig(save_path, dpi=300, bbox_inches="tight")

#     plt.show()

In [ ]:
import math

# Plotting function for NMF topics
def plot_top_words(model, feature_names, n_top_words, title, save_dir=FIGURE_PATH, filename=None):
    n_topics = model.components_.shape[0]

    # Grilla dinámica: intenta mantener proporción similar a la original (2 filas aprox.)
    n_cols = math.ceil(n_topics / 2)
    n_rows = 2 if n_topics > n_cols else 1

    # Ajuste por si con n_cols calculado sobran/faltan casillas
    while n_rows * n_cols < n_topics:
        n_cols += 1

    fig, axes = plt.subplots(n_rows, n_cols, figsize=(6 * n_cols, 7.5 * n_rows), sharex=True)
    axes = np.array(axes).flatten()  # funciona igual si es 1D o 2D

    for topic_idx, topic in enumerate(model.components_):
        top_features_ind = topic.argsort()[-n_top_words:]
        top_features = feature_names[top_features_ind]
        weights = topic[top_features_ind]

        ax = axes[topic_idx]
        ax.barh(top_features, weights, height=0.7)
        ax.set_title(f"Topic {topic_idx + 1}", fontdict={"fontsize": 30})
        ax.tick_params(axis="both", which="major", labelsize=20)
        for i in "top right left".split():
            ax.spines[i].set_visible(False)
        fig.suptitle(title, fontsize=40)

    # Ocultar subplots sobrantes (si la grilla quedó más grande que n_topics)
    for empty_idx in range(n_topics, len(axes)):
        axes[empty_idx].axis("off")

    plt.subplots_adjust(top=0.90, bottom=0.05, wspace=0.90, hspace=0.3)

    save_path = save_dir / filename
    plt.savefig(save_path, dpi=300, bbox_inches="tight")
    plt.show()

# Definición parámetros

In [ ]:
# Parameters for NMF model

n_texts = len(data["obs20_no_stopwords_no_adverbs"])
# n_features: Maximum number of terms to consider in the vocabulary
# The most frequent terms will be kept, and the rest will be discarded.
n_features = 1000
# n_top_words: Number of top words to display for each topic
# used in the plot_top_words function to visualize the topics only
n_top_words = 20
# batch_size: Number of samples to use in each iteration of
# the MiniBatchNMF algorithm
batch_size = 128
# init: Initializer for the NMF algorithm >
# Nonnegative Double Singular Value Decomposition ("a" variant)
init = "nndsvda"

In [ ]:
# n_texts

# TF-IDF

In [ ]:
# Vocabulary real size

# Vectorizador SIN límite de max_features, solo para diagnóstico
# Usa los mismos filtros (max_df, min_df) que el vectorizador real,
# así el conteo refleja el vocabulario que realmente sobrevive a esos filtros
diagnostic_vectorizer = TfidfVectorizer(max_df=0.95, min_df=2)
diagnostic_tfidf = diagnostic_vectorizer.fit_transform(data["obs20_no_stopwords_no_adverbs"])

vocab_size = len(diagnostic_vectorizer.get_feature_names_out())
print(f"Tamaño real del vocabulario (tras max_df=0.95, min_df=2): {vocab_size} palabras únicas")
print(f"n_features actual: {n_features}")

if vocab_size > n_features:
    print(f"\n⚠️  Se están descartando {vocab_size - n_features} palabras "
          f"({(vocab_size - n_features) / vocab_size:.1%} del vocabulario filtrado)")
else:
    print(f"\n✅ n_features={n_features} ya cubre el vocabulario completo, no hay pérdida de información")

In [ ]:
# Frecuencia total de cada palabra en el corpus (suma de ocurrencias, no solo presencia binaria)
# Usamos el CountVectorizer con los mismos filtros, para obtener conteos reales (no pesos TF-IDF)
from sklearn.feature_extraction.text import CountVectorizer

count_vectorizer = CountVectorizer(max_df=0.95, min_df=2)
counts = count_vectorizer.fit_transform(data["obs20_no_stopwords_no_adverbs"])

# Frecuencia total de cada palabra (suma por columna)
word_frequencies = np.asarray(counts.sum(axis=0)).flatten()

# Ordenar de mayor a menor frecuencia
sorted_freqs = np.sort(word_frequencies)[::-1]
total_occurrences = sorted_freqs.sum()

# Cobertura acumulada: para distintos valores de n_features, ¿qué % de las ocurrencias totales cubro?
n_features_candidates = [500, 1000, 1500, 2000, 2500, 3000, 4000, 5000, 6000, 7664]

print(f"Total de ocurrencias en el corpus: {total_occurrences}")
print(f"Vocabulario total (tras filtros): {len(sorted_freqs)} palabras\n")

coverage_by_n = {}
for n in n_features_candidates:
    n = min(n, len(sorted_freqs))  # por si algún candidato excede el vocabulario real
    coverage = sorted_freqs[:n].sum() / total_occurrences
    coverage_by_n[n] = coverage
    print(f"n_features={n:5d} | cobertura de ocurrencias: {coverage:.1%} | palabras descartadas: {len(sorted_freqs) - n}")

In [ ]:
plt.figure(figsize=(10, 6))
plt.plot(list(coverage_by_n.keys()), [v * 100 for v in coverage_by_n.values()], marker="o")
plt.axhline(y=90, color="gray", linestyle="--", alpha=0.5, label="90% de cobertura")
plt.xlabel("n_features")
plt.ylabel("% de ocurrencias totales cubiertas")
plt.title("Cobertura acumulada del vocabulario según n_features")
plt.legend()
plt.grid(True, alpha=0.3)
plt.savefig(FIGURE_PATH / "vocab_coverage_diagnostic.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
# adjust n_features based on the coverage analysis
n_features = 3000  # adjust as needed according to corpus

In [ ]:
# Use tf-idf features for NMF.
print("Extracting tf-idf features for NMF...")
tfidf_vectorizer = TfidfVectorizer(
    max_df=0.95, min_df=2, max_features=n_features
)
t0 = time()
tfidf = tfidf_vectorizer.fit_transform(data["obs20_no_stopwords_no_adverbs"])
print("done in %0.3fs." % (time() - t0))

# Variante 1. NMF Model - Frobenius norm

## Coherencia semántica

In [ ]:
# Semantic coherence score calculation with Gensim's CoherenceModel

from gensim.corpora import Dictionary
from gensim.models import CoherenceModel

# Tokenizamos los textos usando el MISMO analizador que usó el TfidfVectorizer,
# para que la coherencia se calcule sobre el mismo preprocesamiento/vocabulario
# que se usó para entrenar los modelos NMF
analyzer = tfidf_vectorizer.build_analyzer()
texts = [analyzer(doc) for doc in data["obs20_no_stopwords_no_adverbs"]]

# Diccionario gensim (mapeo palabra <-> id), requerido por CoherenceModel
dictionary = Dictionary(texts)

In [ ]:
# Coherence score functions

def get_topics_words(model, feature_names, n_top_words=10):
    """
    Extrae, para cada tópico del modelo NMF, la lista de palabras top (como strings).
    Este es el formato que espera gensim.CoherenceModel (topics=lista de listas de palabras).
    """
    topics_words = []
    for topic in model.components_:
        top_idx = topic.argsort()[-n_top_words:]
        topics_words.append([feature_names[i] for i in top_idx])
    return topics_words


def average_topic_coherence(model, feature_names, texts, dictionary,
                             n_top_words=10, coherence_measure="c_v"):
    """
    Calcula la coherencia promedio de todos los tópicos de un modelo NMF ya entrenado,
    usando gensim.CoherenceModel.
    coherence_measure= "c_v": coherence measure is based on a sliding window,
    a one-set segmentation of the top words and an indirect confirmation
    measure that uses normalized pointwise mutual information (NPMI)
    and the cosine similarity.
    """
    topics_words = get_topics_words(model, feature_names, n_top_words)
    cm = CoherenceModel(
        topics=topics_words,
        texts=texts,
        dictionary=dictionary,
        coherence=coherence_measure,
    )
    return cm.get_coherence()

## Selección `n_components` - parámetro `k`

Antes de entrenar los modelos finales, evaluamos distintos valores de k combinando:
1. Método del codo (error de reconstrucción)
2. Coherencia semántica de tópicos

para elegir el número de tópicos más adecuado (parámetro `k`).

In [ ]:
# Evaluación combinada: error de reconstrucción (codo) + coherencia semántica (c_v), por cada k

k_range = range(2, 16)
reconstruction_errors = []
coherence_scores = []

tfidf_feature_names = tfidf_vectorizer.get_feature_names_out()

for k in k_range:
    t0 = time()
    nmf_temp = NMF(
        n_components=k,
        random_state=1,
        init=init,
        beta_loss="frobenius",
        alpha_W=0.00005,
        alpha_H=0.00005,
        l1_ratio=1,
    ).fit(tfidf)

    error = nmf_temp.reconstruction_err_
    coherence = average_topic_coherence(
        nmf_temp, tfidf_feature_names, texts, dictionary,
        n_top_words=10, coherence_measure="c_v"
    )

    reconstruction_errors.append(error)
    coherence_scores.append(coherence)

    print(f"k={k} | error={error:.4f} | coherencia(c_v)={coherence:.4f} | tiempo={time()-t0:.2f}s")

In [ ]:
# Plot combinado: error de reconstrucción (codo) + coherencia semántica

fig, ax1 = plt.subplots(figsize=(12, 6))

color1 = "tab:blue"
ax1.set_xlabel("Número de tópicos (k)")
ax1.set_ylabel("Error de reconstrucción (Frobenius)", color=color1)
ax1.plot(list(k_range), reconstruction_errors, marker="o", color=color1, label="Error de reconstrucción")
ax1.tick_params(axis="y", labelcolor=color1)
ax1.set_xticks(list(k_range))

ax2 = ax1.twinx()
color2 = "tab:red"
ax2.set_ylabel("Coherencia semántica (c_v)", color=color2)
ax2.plot(list(k_range), coherence_scores, marker="s", color=color2, label="Coherencia (c_v)")
ax2.tick_params(axis="y", labelcolor=color2)

plt.title("Selección de k — Error de reconstrucción vs. Coherencia semántica")
fig.tight_layout()
plt.savefig(FIGURE_PATH / "k_selection_elbow_coherence.png", dpi=300, bbox_inches="tight")
plt.show()

> **Qué es realmente c_v**

c_v es una métrica compuesta — internamente combina varias cosas (co-ocurrencia de palabras en ventanas deslizantes, información mutua puntual normalizada — NPMI, y similitud coseno entre vectores de contexto). No es una probabilidad ni un porcentaje de "aciertos"; es un índice relativo que se interpreta principalmente comparando valores entre sí, no en términos absolutos.

| Rango de c_v	| Interpretación típica |
|---|---|
| < 0.3	| Coherencia baja — palabras del tópico poco relacionadas entre sí |
| 0.3 – 0.5	| Coherencia moderada — común en corpus reales, especialmente con texto corto/informal como encuestas |
| 0.5 – 0.7	| Coherencia buena — tópicos bastante interpretables |
| > 0.7	| Coherencia muy alta — poco común fuera de corpus muy limpios/grandes/especializados |

Coherencia alta con k muy bajo es una trampa común. Con solo 2-4 tópicos, es fácil que la coherencia salga alta simplemente porque los tópicos son muy amplios/genéricos. La métrica premia esa amplitud, pero pierde la capacidad de diferenciar temas específicos.

> **Para este caso**

Se observan dos picos del puntaje de coherencia:
- 4 tópicos
- 13 a 14 tópicos -> se elige el valor de 14 tópicos para tener más granularidad temática

## Elección de k_base

In [ ]:
# Elección de k_base a partir del gráfico combinado (codo + coherencia)
#
# Criterio: punto donde la coherencia (c_v) alcanza un máximo o meseta alta,
# preferentemente cerca de donde el error de reconstrucción empieza a aplanarse.
#
# Este valor se define manualmente tras inspeccionar el gráfico anterior.
# Actualizar k_base con el valor elegido antes de continuar.

k_base = 14  # ← reemplazar con el valor de k elegido tras ver el gráfico

assert k_base is not None, "Debes definir k_base tras inspeccionar el gráfico de codo/coherencia"
assert k_base in k_range, f"k_base debe estar dentro del rango evaluado {list(k_range)}"

print(f"k_base seleccionado: {k_base}")
print(f"  Error de reconstrucción en k_base: {reconstruction_errors[list(k_range).index(k_base)]:.4f}")
print(f"  Coherencia (c_v) en k_base: {coherence_scores[list(k_range).index(k_base)]:.4f}")

## Fit NMF Model - Frobenius norm

In [ ]:
# Fit the NMF model - Frobenius norm
print(
    "Fitting the NMF model (Frobenius norm) with tf-idf features, "
    "n_texts=%d and n_features=%d..." % (n_texts, n_features)
)
t0 = time()
nmf = NMF(
    n_components=k_base,
    random_state=1,
    init=init,
    beta_loss="frobenius",
    alpha_W=0.00005,
    alpha_H=0.00005,
    l1_ratio=1,
).fit(tfidf)
print("done in %0.3fs." % (time() - t0))

In [ ]:
# Plot the top words for each topic in the NMF model - Frobenius norm
tfidf_feature_names = tfidf_vectorizer.get_feature_names_out()
plot_top_words(
    nmf, tfidf_feature_names, n_top_words, "Topics in NMF model (Frobenius norm)",
    filename="topics_nmf_frobenius.png"
)

# Variante 2. NMF Model - Kullback-Leibler divergence

## Fit NMF Model - Kullback-Leibler divergence

In [ ]:
# Fit NMF Model - Kullback-Leibler divergence

print(
    "\n" * 2,
    "Fitting the NMF model (generalized Kullback-Leibler "
    "divergence) with tf-idf features, n_texts=%d and n_features=%d..."
    % (n_texts, n_features),
)
t0 = time()
nmf = NMF(
    n_components=k_base,
    random_state=1,
    init=init,
    beta_loss="kullback-leibler",
    solver="mu",
    max_iter=1000,
    alpha_W=0.00005,
    alpha_H=0.00005,
    l1_ratio=0.5,
).fit(tfidf)
print("done in %0.3fs." % (time() - t0))

In [ ]:
# Plot the top words for each topic in the NMF model -
# generalized Kullback-Leibler divergence
tfidf_feature_names = tfidf_vectorizer.get_feature_names_out()
plot_top_words(
    nmf,
    tfidf_feature_names,
    n_top_words,
    "Topics in NMF model (generalized Kullback-Leibler divergence)",
    filename="topics_nmf_kullback-leibler.png"
)

# Variante 3. the MiniBatchNMF model - Frobenius norm

## Fit the MiniBatchNMF model - Frobenius norm

In [ ]:
# Fit the MiniBatchNMF model - Frobenius norm

print(
    "\n" * 2,
    "Fitting the MiniBatchNMF model (Frobenius norm) with tf-idf "
    "features, n_texts=%d and n_features=%d, batch_size=%d..."
    % (n_texts, n_features, batch_size),
)
t0 = time()
mbnmf = MiniBatchNMF(
    n_components=k_base,
    random_state=1,
    batch_size=batch_size,
    init=init,
    beta_loss="frobenius",
    alpha_W=0.00005,
    alpha_H=0.00005,
    l1_ratio=0.5,
).fit(tfidf)
print("done in %0.3fs." % (time() - t0))

In [ ]:
# Plot the top words for each topic in the MiniBatchNMF model -
# Frobenius norm
tfidf_feature_names = tfidf_vectorizer.get_feature_names_out()
plot_top_words(
    mbnmf,
    tfidf_feature_names,
    n_top_words,
    "Topics in MiniBatchNMF model (Frobenius norm)",
    filename="topics_minibatchnmf_frobenius.png"
)

# Variante 4. the MiniBatchNMF model - generalized Kullback-Leibler

## Fit the MiniBatchNMF model - generalized Kullback-Leibler divergence

In [ ]:
# Fit the MiniBatchNMF model - generalized Kullback-Leibler divergence

print(
    "\n" * 2,
    "Fitting the MiniBatchNMF model (generalized Kullback-Leibler "
    "divergence) with tf-idf features, n_texts=%d and n_features=%d, "
    "batch_size=%d..." % (n_texts, n_features, batch_size),
)
t0 = time()
mbnmf = MiniBatchNMF(
    n_components=k_base,
    random_state=1,
    batch_size=batch_size,
    init=init,
    beta_loss="kullback-leibler",
    alpha_W=0.00005,
    alpha_H=0.00005,
    l1_ratio=0.5,
).fit(tfidf)
print("done in %0.3fs." % (time() - t0))

In [ ]:
# Plot the top words for each topic in the MiniBatchNMF model -
# generalized Kullback-Leibler divergence

tfidf_feature_names = tfidf_vectorizer.get_feature_names_out()
plot_top_words(
    mbnmf,
    tfidf_feature_names,
    n_top_words,
    "Topics in MiniBatchNMF model (generalized Kullback-Leibler divergence)",
    filename="topics_minibatchnmf_kullback-leibler.png"
)

# Refinamiento de k para la variante ganadora

## Selección de la variante ganadora

Tras inspeccionar visualmente los 4 conjuntos de tópicos (plot_top_words de cada variante),
se elige la variante con tópicos más interpretables/menos solapados, y se refina k
específicamente para ella.

In [ ]:
# Definición de las 4 variantes posibles (mismos hiperparámetros que la Etapa 1)
# Elegir UNA como ganadora tras ver los resultados de plot_top_words

VARIANT_CONFIGS = {
    "nmf_frobenius": {
        "model_class": NMF,
        "params": dict(
            random_state=1, init=init, beta_loss="frobenius",
            alpha_W=0.00005, alpha_H=0.00005, l1_ratio=1,
        ),
    },
    "nmf_kl": {
        "model_class": NMF,
        "params": dict(
            random_state=1, init=init, beta_loss="kullback-leibler",
            solver="mu", max_iter=1000,
            alpha_W=0.00005, alpha_H=0.00005, l1_ratio=0.5,
        ),
    },
    "minibatch_frobenius": {
        "model_class": MiniBatchNMF,
        "params": dict(
            random_state=1, batch_size=batch_size, init=init, beta_loss="frobenius",
            alpha_W=0.00005, alpha_H=0.00005, l1_ratio=0.5,
        ),
    },
    "minibatch_kl": {
        "model_class": MiniBatchNMF,
        "params": dict(
            random_state=1, batch_size=batch_size, init=init, beta_loss="kullback-leibler",
            alpha_W=0.00005, alpha_H=0.00005, l1_ratio=0.5,
        ),
    },
}

# ← Elegir tras inspeccionar visualmente los 4 resultados de plot_top_words
winning_variant = "nmf_frobenius"  # opciones: "nmf_frobenius", "nmf_kl", "minibatch_frobenius", "minibatch_kl"


Definir variable `winning_variant`

In [ ]:
assert winning_variant in VARIANT_CONFIGS, (
    f"Debes definir winning_variant como una de: {list(VARIANT_CONFIGS.keys())}"
)

print(f"Variante ganadora seleccionada: {winning_variant}")

Refinamiento

In [ ]:
# Refinamiento: mismo procedimiento del Cambio 4, pero usando SOLO la variante ganadora
# Rango de búsqueda centrado en k_base, para no repetir todo el rango original
k_range_refine = range(max(2, k_base - 3), k_base + 4)  # ajustable

refine_reconstruction_errors = []
refine_coherence_scores = []

variant_class = VARIANT_CONFIGS[winning_variant]["model_class"]
variant_params = VARIANT_CONFIGS[winning_variant]["params"]

for k in k_range_refine:
    t0 = time()
    model_temp = variant_class(n_components=k, **variant_params).fit(tfidf)

    error = model_temp.reconstruction_err_
    coherence = average_topic_coherence(
        model_temp, tfidf_feature_names, texts, dictionary,
        n_top_words=10, coherence_measure="c_v"
    )

    refine_reconstruction_errors.append(error)
    refine_coherence_scores.append(coherence)

    print(f"[{winning_variant}] k={k} | error={error:.4f} | coherencia(c_v)={coherence:.4f} | tiempo={time()-t0:.2f}s")

In [ ]:
# Plot combinado del refinamiento (mismo formato del Cambio 4)

fig, ax1 = plt.subplots(figsize=(12, 6))

color1 = "tab:blue"
ax1.set_xlabel("Número de tópicos (k)")
ax1.set_ylabel("Error de reconstrucción", color=color1)
ax1.plot(list(k_range_refine), refine_reconstruction_errors, marker="o", color=color1)
ax1.tick_params(axis="y", labelcolor=color1)
ax1.set_xticks(list(k_range_refine))

ax2 = ax1.twinx()
color2 = "tab:red"
ax2.set_ylabel("Coherencia semántica (c_v)", color=color2)
ax2.plot(list(k_range_refine), refine_coherence_scores, marker="s", color=color2)
ax2.tick_params(axis="y", labelcolor=color2)

plt.title(f"Refinamiento de k — variante ganadora: {winning_variant}")
fig.tight_layout()
plt.savefig(FIGURE_PATH / f"k_refinement_{winning_variant}.png", dpi=300, bbox_inches="tight")
plt.show()

## Elección de k_final

In [ ]:
k_final = 14  # ← reemplazar con el valor elegido tras ver el gráfico de refinamiento

assert k_final is not None, "Debes definir k_final tras inspeccionar el gráfico de refinamiento"
assert k_final in k_range_refine, f"k_final debe estar dentro del rango refinado {list(k_range_refine)}"

print(f"k_final seleccionado ({winning_variant}): {k_final}")

# Modelo final

## Entrenamiento del modelo final

In [ ]:
# Entrenamiento del modelo final: variante ganadora + k_final ya elegidos
final_model = variant_class(n_components=k_final, **variant_params).fit(tfidf)

print(f"Modelo final entrenado: {winning_variant} | k_final={k_final}")

## Visualización de tópicos del modelo final

In [ ]:
# Reutilizamos plot_top_words, ya definida al inicio del notebook
plot_top_words(
    final_model,
    tfidf_feature_names,
    n_top_words,
    f"Topics in final model ({winning_variant}, k={k_final})",
    filename=f"topics_final_{winning_variant}_k{k_final}.png"
)

## Automatización de nombres de tópicos

In [ ]:
def name_topics(model, feature_names, n_words_for_name=2):
    """
    Asigna un nombre automático a cada tópico, usando la(s) palabra(s)
    de mayor peso en la matriz H (model.components_).

    n_words_for_name: cuántas palabras top usar para construir el nombre
                       (1 = solo la palabra dominante, ej. "excelente";
                        2+ = combinación, ej. "excelente_parece")
    """
    topic_names = {}
    for topic_idx, topic in enumerate(model.components_):
        top_idx = topic.argsort()[-n_words_for_name:][::-1]  # de mayor a menor peso
        top_words = [feature_names[i] for i in top_idx]
        topic_names[topic_idx] = "_".join(top_words)
    return topic_names


def check_duplicate_names(topic_names):
    """
    Detecta si dos o más tópicos terminaron con el mismo nombre automático.
    Retorna un diccionario {nombre: [lista de topic_idx]} solo para los nombres duplicados.
    """
    name_to_topics = {}
    for topic_idx, name in topic_names.items():
        name_to_topics.setdefault(name, []).append(topic_idx)

    duplicates = {name: idxs for name, idxs in name_to_topics.items() if len(idxs) > 1}
    return duplicates


# Intento inicial con 1 palabra
n_words_for_name = 1
topic_names = name_topics(final_model, tfidf_feature_names, n_words_for_name=n_words_for_name)
duplicates = check_duplicate_names(topic_names)

# Si hay duplicados, aumentar automáticamente el número de palabras hasta resolverlos
# (o hasta un máximo razonable, para evitar nombres eternos)
max_words_for_name = 4
while duplicates and n_words_for_name < max_words_for_name:
    n_words_for_name += 1
    topic_names = name_topics(final_model, tfidf_feature_names, n_words_for_name=n_words_for_name)
    duplicates = check_duplicate_names(topic_names)

if duplicates:
    print(f"⚠️  Persisten nombres duplicados incluso con n_words_for_name={n_words_for_name}:")
    for name, idxs in duplicates.items():
        topics_str = ", ".join(f"Topic {i + 1}" for i in idxs)
        print(f"   '{name}' → {topics_str}")
    print("   Revisar manualmente estos tópicos.")
else:
    print(f"✅ Nombres únicos logrados con n_words_for_name={n_words_for_name}\n")

for idx, name in topic_names.items():
    print(f"Topic {idx + 1}: {name}")

# Asignación de tópicos por filas

Objetivo: Asignar un máximo de 5 tópicos a cada fila, según su importancia en la matriz `H`.

## Diagnóstico: distribución de pesos por fila (antes de fijar el umbral)

No todas las filas tendrán 5 tópicos. Para ello se busca definir un umbral de importancia/peso del texto.

In [ ]:
# import numpy as np
# import pandas as pd

# W: matriz filas x tópicos, con el peso (sin normalizar) de cada tópico en cada fila
W = final_model.transform(tfidf)

# Normalizamos cada fila para que sus pesos sumen 1 (proporciones),
# tal como se discutió: así "15%" significa "15% del peso total de ESA fila"
row_sums = W.sum(axis=1, keepdims=True)
row_sums[row_sums == 0] = 1  # evita división por cero en filas sin señal (todas las columnas en 0)
W_normalized = W / row_sums

# Para distintos umbrales candidatos, contamos cuántos tópicos por fila los superan
thresholds_to_test = [0.05, 0.08, 0.10, 0.15, 0.20]

print(f"Total de filas evaluadas: {W_normalized.shape[0]}")
print(f"k_final (número de tópicos): {W_normalized.shape[1]}\n")

diagnostic_summary = {}

for thresh in thresholds_to_test:
    topics_per_row = (W_normalized >= thresh).sum(axis=1)
    diagnostic_summary[thresh] = topics_per_row

    print(f"--- Umbral: {thresh:.0%} ---")
    print(f"  Promedio de tópicos/fila: {topics_per_row.mean():.2f}")
    print(f"  Mediana de tópicos/fila:  {np.median(topics_per_row):.1f}")
    print(f"  Distribución (0 a 5+ tópicos):")
    for n_topics in range(0, 6):
        if n_topics < 5:
            count = (topics_per_row == n_topics).sum()
        else:
            count = (topics_per_row >= 5).sum()
        pct = count / len(topics_per_row)
        label = f"{n_topics}" if n_topics < 5 else "5+"
        print(f"    {label} tópicos: {count:5d} filas ({pct:.1%})")
    print()

In [ ]:
# Visualización: promedio de tópicos asignados por fila, según umbral
avg_topics_by_threshold = [diagnostic_summary[t].mean() for t in thresholds_to_test]

plt.figure(figsize=(10, 6))
plt.plot([f"{t:.0%}" for t in thresholds_to_test], avg_topics_by_threshold, marker="o")
plt.axhline(y=5, color="gray", linestyle="--", alpha=0.5, label="Máximo (5 tópicos)")
plt.xlabel("Umbral de peso mínimo")
plt.ylabel("Promedio de tópicos asignados por fila")
plt.title("Impacto del umbral en la cantidad de tópicos asignados por fila")
plt.legend()
plt.grid(True, alpha=0.3)
plt.savefig(FIGURE_PATH / "threshold_diagnostic.png", dpi=300, bbox_inches="tight")
plt.show()

## Asignación de tópicos a nivel fila

In [ ]:
# Umbral de peso mínimo para que un tópico se considere relevante en una fila
# (ajustar según el diagnóstico anterior)
TOPIC_THRESHOLD = 0.10
MAX_TOPICS_PER_ROW = 5

def assign_row_topics(W_normalized, topic_names, threshold=TOPIC_THRESHOLD, max_topics=MAX_TOPICS_PER_ROW):
    """
    Para cada fila, devuelve los nombres de los tópicos cuyo peso normalizado
    supera 'threshold', ordenados de mayor a menor peso, con un máximo de 'max_topics'.
    """
    row_topics = []

    for row_weights in W_normalized:
        # índices ordenados de mayor a menor peso
        sorted_idx = np.argsort(row_weights)[::-1]

        selected = []
        for idx in sorted_idx:
            if row_weights[idx] >= threshold and len(selected) < max_topics:
                selected.append(topic_names[idx])
            else:
                break  # ya no hay más tópicos que superen el umbral (están ordenados)

        row_topics.append(", ".join(selected))

    return row_topics


data["topicos_nmf"] = assign_row_topics(W_normalized, topic_names, TOPIC_THRESHOLD, MAX_TOPICS_PER_ROW)

# Reemplazar filas sin ningún tópico por encima del umbral con la etiqueta "SIN_TOPICO"
data["topicos_nmf"] = data["topicos_nmf"].replace("", "SIN_TOPICO")

# Vista rápida del resultado
data[["obs20_no_stopwords_no_adverbs", "topicos_nmf"]].head(10)

In [ ]:
# Filas donde ningún tópico superó el umbral (celda 'topicos_nmf' vacía)
empty_topic_rows = data[data["topicos_nmf"] == "SIN_TOPICO"]
print(f"Filas sin tópicos asignados (umbral={TOPIC_THRESHOLD:.0%}): {len(empty_topic_rows)} de {len(data)}")

if len(empty_topic_rows) > 0:
    empty_topic_rows[["obs20_no_stopwords_no_adverbs", "topicos_nmf"]].head(10)

# Guardado en Excel

In [ ]:
# Guardar el Excel con la nueva columna 'topicos_nmf'
output_path = _DATA / "processed" / f"obs20_con_topicos_{winning_variant}_k{k_final}.xlsx"
data.to_excel(output_path, index=False)
print(f"Archivo guardado en: {output_path}")

# Diagnóstico dirigido: por qué una fila específica queda como SIN_TOPICO

> **Filas sin tópico asignado (SIN_TOPICO): 25.5% del corpus (4.239 de 16.632 respuestas)**

Estas respuestas no fueron forzadas a ninguna de las 14 categorías temáticas identificadas por el modelo NMF. El análisis dirigido de casos representativos confirmó que esto no se debe a errores técnicos (vocabulario insuficiente o problemas de indexación), sino a que estas respuestas abordan ***temas minoritarios o muy específicos, presentes en menos del 0.2% del corpus*** (ej. integridad académica, uso indebido de dispositivos móviles, equidad en evaluaciones). La regularización L1 del modelo (`alpha_W`, `l1_ratio`) evita asignarles un tópico de forma artificial cuando ningún patrón dominante los representa adecuadamente. ***Estas respuestas constituyen, en sí mismas, un grupo de interés cualitativo: representan las preocupaciones más singulares o atípicas de los estudiantes, no capturadas por las categorías temáticas mayoritarias***.

In [ ]:
def diagnose_row(row_idx, data, tfidf, tfidf_vectorizer, W, W_normalized,
                  topic_names, text_column="obs20_no_stopwords_no_adverbs"):
    """
    Diagnóstico dirigido para una fila específica:
    - Qué palabras del texto sobrevivieron al vocabulario (Hipótesis 2)
    - Pesos W (crudos y normalizados) en cada tópico (Hipótesis 1)
    - En cuántos documentos del corpus aparece cada palabra clave (Hipótesis 3)
    """
    text = data.iloc[row_idx][text_column]
    vocab = tfidf_vectorizer.vocabulary_  # dict: palabra -> índice de columna
    analyzer = tfidf_vectorizer.build_analyzer()
    tokens = analyzer(text)

    print(f"=== Fila {row_idx} ===")
    print(f"Texto original:\n{text}\n")

    # --- Hipótesis 2: ¿qué palabras sobrevivieron al vocabulario de n_features? ---
    tokens_unique = list(dict.fromkeys(tokens))  # únicos, preservando orden
    survived = [t for t in tokens_unique if t in vocab]
    dropped = [t for t in tokens_unique if t not in vocab]

    print(f"--- Hipótesis 2: cobertura de vocabulario ---")
    print(f"Palabras únicas en el texto: {len(tokens_unique)}")
    print(f"  Sobrevivieron al vocabulario (n_features): {len(survived)} -> {survived}")
    print(f"  Descartadas (fuera de n_features):         {len(dropped)} -> {dropped}\n")

    # --- Hipótesis 3: frecuencia documental de las palabras clave ---
    print(f"--- Hipótesis 3: frecuencia documental de cada palabra sobreviviente ---")
    # document frequency = en cuántas filas del corpus aparece cada palabra (columna binarizada)
    binary_tfidf = (tfidf > 0).astype(int)
    for word in survived:
        col_idx = vocab[word]
        doc_freq = binary_tfidf[:, col_idx].sum()
        pct = doc_freq / tfidf.shape[0]
        print(f"  '{word}': aparece en {doc_freq} de {tfidf.shape[0]} filas ({pct:.2%})")
    print()

    # --- Hipótesis 1: distribución de pesos W para esta fila ---
    print(f"--- Hipótesis 1: distribución de pesos W (crudo y normalizado) ---")
    raw_weights = W[row_idx]
    norm_weights = W_normalized[row_idx]
    sorted_idx = np.argsort(norm_weights)[::-1]

    for idx in sorted_idx:
        print(f"  {topic_names[idx]:30s} | peso crudo: {raw_weights[idx]:.4f} | "
              f"peso normalizado: {norm_weights[idx]:.2%}")

    print(f"\n  Suma total de pesos crudos (peso total asignable): {raw_weights.sum():.4f}")
    print(f"  Peso máximo normalizado: {norm_weights.max():.2%} "
          f"(umbral actual: {TOPIC_THRESHOLD:.0%})")


# Buscar la fila del ejemplo que compartiste (ajustar el texto de búsqueda si no coincide exacto)
example_mask = data["obs20_no_stopwords_no_adverbs"].str.contains("integridad academica evaluaciones evidenciado casos uso indebido dispositivos moviles", na=False)
example_idx = data[example_mask].index

if len(example_idx) > 0:
    diagnose_row(example_idx[0], data, tfidf, tfidf_vectorizer, W, W_normalized, topic_names)
else:
    print("No se encontró la fila exacta — ajustar el texto de búsqueda en 'example_mask'")

In [ ]:
row_idx = example_idx[0]
print(f"row_idx recuperado: {row_idx}")

# Verificación: ¿el vector TF-IDF de esta fila es realmente cero?
row_tfidf_sum = tfidf[row_idx].sum()
print(f"Suma del vector TF-IDF para row_idx={row_idx}: {row_tfidf_sum:.4f}")

# Verificación: ¿data.index es un RangeIndex contiguo (0, 1, 2, ..., n-1)?
is_default_index = data.index.equals(pd.RangeIndex(len(data)))
print(f"¿data.index es el RangeIndex por defecto (sin huecos)? {is_default_index}")

if not is_default_index:
    print("⚠️  El índice de 'data' NO es el rango por defecto — esto puede causar el desajuste.")
    print(f"   Ejemplo: primeras etiquetas del índice: {data.index[:10].tolist()}")

In [ ]:
print(f"tfidf.shape: {tfidf.shape}")
print(f"W.shape: {W.shape}")
print(f"final_model.components_.shape: {final_model.components_.shape}")
print(f"final_model n_components: {final_model.n_components}")

# ¿El número de columnas de tfidf coincide con lo que espera final_model?
print(f"\n¿tfidf y final_model son compatibles? "
      f"{tfidf.shape[1] == final_model.components_.shape[1]}")

# ¿W tiene tantas filas como tfidf?
print(f"¿W y tfidf tienen el mismo número de filas? {W.shape[0] == tfidf.shape[0]}")

In [ ]:
print("Valores crudos de W para esta fila (precisión completa):")
print(repr(W[row_idx]))

print(f"\n¿Todos son exactamente 0.0? {np.all(W[row_idx] == 0.0)}")
print(f"Valor máximo: {W[row_idx].max():.10f}")
print(f"Valor mínimo: {W[row_idx].min():.10f}")

In [ ]:
print(f"Valor en data['topicos_nmf'] para la fila {row_idx}: {data.loc[row_idx, 'topicos_nmf']!r}")